# Embeddings, Clustering, and Theme Extraction

An embedding converts an item into a vector so that semantically related items can be compared or grouped.


## What to learn

- Similarity search finds nearby vectors for retrieval or deduplication.
- Clustering groups items without predefined labels.
- An LLM can name and explain the resulting groups.
- Human review checks that themes are coherent and useful.

## Decision rule

Use embeddings for candidate generation and scale; use language models or people for interpretation. Evaluate cluster stability, inspect representative and borderline examples, and do not treat distance as ground truth.


## Runnable example

Run the following cells, then change one input and observe what changes. Focus on the decision being demonstrated rather than framework-specific syntax.


In [ ]:
"""Tiny end-to-end theme pipeline: embed -> cluster -> name -> classify.

Uses toy 2-D "embeddings" so it runs anywhere; the structure is exactly the
production pipeline (swap in real embeddings + HDBSCAN + LLM calls).
"""
# Import the dependencies used by this example.
import math
from collections import defaultdict

# Toy corpus: utterances with hand-set 2-D embeddings (real: 1024-3072 dims)
UTTERANCES = [
    ("u1", "The signup flow kept erroring out", (0.9, 0.1)),
    ("u2", "I could not finish creating my account", (0.85, 0.15)),
    ("u3", "Onboarding took me three tries", (0.8, 0.2)),
    ("u4", "The price is way too high for what you get", (0.1, 0.9)),
    ("u5", "I don't understand what the tiers include", (0.15, 0.85)),
    ("u6", "Support never answered my email", (0.5, 0.5)),
]


# Define the data shape and small operations before running them.
def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb)


# --- 1. Cluster (toy: greedy threshold; production: UMAP + HDBSCAN) ------
def cluster(items, threshold=0.95):
    clusters = []
    for uid, text, vec in items:
        for c in clusters:
            if cosine(vec, c["centroid"]) >= threshold:
                c["members"].append((uid, text))
                break
        else:
            clusters.append({"centroid": vec, "members": [(uid, text)]})
    return clusters


clusters = cluster(UTTERANCES)

# --- 2. Name clusters (production: LLM given representative docs) --------
NAMES = {0: "onboarding friction", 1: "pricing confusion", 2: "support responsiveness"}
# .get() fallback: a new cluster (e.g. from Practice Q1) lands in an
# "unnamed" bucket instead of crashing — mirroring the real 'uncategorized'
# bucket a production taxonomy needs.
taxonomy = {NAMES.get(i, f"unnamed-cluster-{i}"): [uid for uid, _ in c["members"]]
            for i, c in enumerate(clusters)}

# --- 3. Classify everything against the frozen taxonomy -> exact counts --
counts = {theme: len(uids) for theme, uids in taxonomy.items()}
print("taxonomy v1 (frozen):")
for theme, uids in taxonomy.items():
    print(f"  {theme}: n={len(uids)}, evidence={uids}")

# Small clusters are 'emerging signals', not established themes
for theme, n in counts.items():
    tier = "established" if n >= 3 else "emerging signal (low n)"
    print(f"  -> {theme}: {tier}")


## Study questions

1. What problem does this pattern solve?
2. What is the simplest alternative?
3. Which failure would tell you that the pattern is implemented incorrectly?
4. What measurement would justify adding more complexity?

## Before using it

- [ ] The requirement and success measure are explicit.
- [ ] Inputs and outputs are validated.
- [ ] Failures, retries, and stopping conditions are bounded.
- [ ] Security and privacy match the consequences of the action.
- [ ] The behavior is tested on realistic examples.
